# 1.8 Python

Python is a powerful, widely used programming language for scientific computing, data analysis, and statistics, freely available for Mac OS X, Windows, and UNIX systems. Knowing how to use Python is an extremely useful skill. Python and various supporting information can be obtained at https://www.python.org. The Anaconda distribution (https://www.anaconda.com) bundles Python together with the scientific libraries used throughout these notes, and is an excellent way to get started.

In the Python section at the end of each chapter, we provide Python code to let you try out some of the examples from the chapter, especially via simulation. These sections are not intended to be a full introduction to Python; many Python tutorials are available for free online. But the Python sections show how to implement various simulations, computations, and visualizations that naturally accompany the material from each chapter.

We rely on three libraries throughout:

- **NumPy** (`numpy`) — arrays and numerical operations  
- **SciPy** (`scipy`) — scientific functions including special mathematical functions  
- **Matplotlib** (`matplotlib`) — plotting and visualization

The first step in any Python session is to import these libraries.

In [ ]:
import math
import numpy as np
import scipy.special as sp
import matplotlib.pyplot as plt

## Arrays

NumPy is built around *arrays*, and getting familiar with "vectorized thinking" is very important for using Python effectively for statistics. To create an array, we use `np.array` and pass a list of values. For example,

In [ ]:
v = np.array([3, 1, 4, 1, 5, 9])
v

defines `v` to be the array (3, 1, 4, 1, 5, 9). Similarly, `n = 110` sets `n` equal to 110; Python views `n` as a plain integer (a scalar).

In [ ]:
n = 110
print(v.sum())    # sum of all entries
print(v.max())    # largest value
print(v.min())    # smallest value
print(len(v))     # number of entries

`v.sum()` adds up the entries of `v`, `v.max()` gives the largest value, `v.min()` gives the smallest value, and `len(v)` gives the length.

A convenient way to get the array (1, 2, …, n) is `np.arange(1, n+1)`. Note that `np.arange(a, b)` produces integers from `a` up to *but not including* `b`, so we write `n+1` to include `n` itself. More generally, `np.arange(m, n+1)` gives the sequence of integers from `m` to `n`.

In [ ]:
n = 10
np.arange(1, n + 1)    # the array (1, 2, ..., 10)

To access the *i*-th entry of an array `v`, use `v[i-1]`. Python uses **0-based indexing**, meaning the first element is at position 0, the second at position 1, and so on. So the first entry of `v` is `v[0]`, the second is `v[1]`, etc. We can also extract several entries at once by passing a list of indices:

In [ ]:
v = np.array([3, 1, 4, 1, 5, 9])
print(v[0])              # first entry
print(v[[0, 2, 4]])      # 1st, 3rd, and 5th entries

gives the array consisting of the 1st, 3rd, and 5th entries of `v`. It is also possible to get a subarray by specifying which indices to *remove*, using `np.delete`:

In [ ]:
np.delete(v, [1, 2, 3])  # remove the 2nd, 3rd, and 4th entries (indices 1, 2, 3)

gives the array obtained by removing the 2nd through 4th entries of `v`.

Many operations in Python with NumPy are interpreted **element-wise**. For example, `v**3` cubes each entry individually:

In [ ]:
v**3

Similarly,

In [ ]:
1 / np.arange(1, 101)**2

is a very compact way to get the array (1, 1/4, 1/9, …, 1/10000). Arithmetic between an array and a scalar applies the operation to every element — for example, `v + 3` adds 3 to each entry of `v`:

In [ ]:
v + 3

## Factorials and Binomial Coefficients

We can compute $n!$ using `math.factorial(n)` and $\binom{n}{k}$ using `math.comb(n, k)`. As we have seen, factorials grow extremely quickly. Unlike many languages, Python's `math.factorial` uses exact integer arithmetic and never overflows — it will happily compute `factorial(1000)` and return the exact answer. For very large $n$ where we only need the *logarithm* of the factorial (which is common in probability calculations), we can use `math.lgamma(n+1)`, which computes $\log(n!)$. Similarly, `sp.gammaln(n+1) - sp.gammaln(k+1) - sp.gammaln(n-k+1)` computes $\log\binom{n}{k}$.

In [ ]:
math.factorial(10)      # 10!

In [ ]:
math.comb(10, 3)        # C(10, 3)

In [ ]:
# Log-factorial and log-binomial coefficient for large values
n, k = 1000, 500
print(math.lgamma(n + 1))                                                     # log(1000!)
print(sp.gammaln(n+1) - sp.gammaln(k+1) - sp.gammaln(n - k + 1))            # log C(1000, 500)

## Sampling and Simulation

The `numpy.random` module provides a variety of tools for drawing random samples. (Technically they are pseudo-random since there is an underlying deterministic algorithm, but they look like random samples for almost all practical purposes.) We create a *random number generator* object once and use it throughout:

In [ ]:
rng = np.random.default_rng(seed=42)   # seed makes results reproducible

`rng.choice(population, k, replace=False)` draws an ordered random sample of `k` elements from `population`, without replacement, with equal probability given to each element. For example,

In [ ]:
n, k = 10, 5
rng.choice(np.arange(1, n + 1), k, replace=False)

generates an ordered random sample of 5 of the numbers from 1 to 10, without replacement. To sample *with* replacement instead, set `replace=True`:

In [ ]:
rng.choice(np.arange(1, n + 1), k, replace=True)

To generate a random permutation of 1, 2, …, n we can use `rng.permutation`:

In [ ]:
rng.permutation(np.arange(1, n + 1))   # random permutation of 1..n

We can also use `rng.choice` to draw from a non-numeric array. Python's `string.ascii_lowercase` is the string of the 26 lowercase letters of the English alphabet. Turning it into a NumPy array and sampling from it gives a random "word":

In [ ]:
import string
letters = np.array(list(string.ascii_lowercase))   # the 26 lowercase letters
''.join(rng.choice(letters, 7, replace=False))      # random 7-letter "word"

`rng.choice` also allows us to specify general probabilities for sampling each element. For example,

In [ ]:
rng.choice(np.arange(1, 5), 3, replace=True, p=[0.1, 0.2, 0.3, 0.4])

samples three numbers from {1, 2, 3, 4}, with replacement, with probabilities given by (0.1, 0.2, 0.3, 0.4). If the sampling is without replacement, then at each stage the probability of any not-yet-chosen element is proportional to its original probability.

Generating many random samples allows us to perform a **simulation** for a probability problem. A *list comprehension* of the form

```python
[expression for _ in range(N)]
```

evaluates `expression` exactly `N` times and collects the results in a list — this is the standard Python idiom for repeating a random experiment many times.

## Matching Problem Simulation

Let's show by simulation that the probability of a matching card in Example 1.6.4 is approximately $1 - 1/e$ when the deck is sufficiently large. We perform the experiment many times and count how often we get at least one matching card:

In [ ]:
n = 100   # number of cards
r = [np.sum(rng.permutation(n) == np.arange(n)) for _ in range(10**4)]   # shuffle; count matches
np.sum(np.array(r) >= 1) / 10**4   # proportion with at least one match

In the first line, we choose how many cards are in the deck (here, 100 cards). In the second line, let's work from the inside out:

- `rng.permutation(n)` produces a random shuffle of the integers 0, 1, …, n−1. We think of position `i` as the "correct" slot for card `i`.
- `rng.permutation(n) == np.arange(n)` is a Boolean array of length `n`, the `i`-th element of which is `True` if card `i` ended up in position `i`, and `False` otherwise. `np.sum(...)` counts the `True` values, giving us the number of matching cards in this run.
- The list comprehension repeats this experiment 10 000 times, storing the match counts in `r`.

In the last line, we count how often there was at least one match and divide by the number of simulations.

To explain what the code is doing within the code itself rather than in separate documentation, we can add comments using the `#` symbol. Comments are ignored by Python but make the code much easier to understand. A commented version of the simulation above is shown below, together with a comparison to the theoretical value:

In [ ]:
n = 100  # number of cards
r = [np.sum(rng.permutation(n) == np.arange(n))   # shuffle; count matches
     for _ in range(10**4)]
simulated = np.sum(np.array(r) >= 1) / 10**4      # proportion with a match

print(f'Simulated:   {simulated:.4f}')
print(f'1 - 1/e   :  {1 - 1/math.e:.4f}')

What did you get when you ran the code? We got approximately 0.63, which is quite close to $1 - 1/e$.

## Birthday Problem Calculation and Simulation

The following code uses `np.prod` (which gives the product of an array) to calculate the probability of at least one birthday match in a group of 23 people:

In [ ]:
k = 23
1 - np.prod(np.arange(365 - k + 1, 366)) / 365**k

Better yet, we can write reusable functions `pbirthday` and `qbirthday` that work exactly like the built-in birthday functions found in statistics packages:

- `pbirthday(k)` returns the probability of at least one shared birthday if the room has `k` people.
- `qbirthday(p)` returns the number of people needed in order to have probability `p` of at least one shared birthday.

In [ ]:
def pbirthday(k, coincident=2, classes=365):
    """P(at least `coincident` people share a birthday) with `k` people."""
    if coincident == 2:
        p_all_distinct = np.prod(np.arange(classes - k + 1, classes + 1)) / classes**k
        return 1 - p_all_distinct
    else:
        # For higher coincidences we use simulation
        sim_rng = np.random.default_rng(0)
        trials = sim_rng.integers(0, classes, size=(50_000, k))
        max_counts = np.array([np.bincount(row).max() for row in trials])
        return np.mean(max_counts >= coincident)

def qbirthday(p, coincident=2, classes=365):
    """Smallest k such that P(at least one shared birthday) >= p."""
    for k in range(2, classes + 1):
        if pbirthday(k, coincident=coincident, classes=classes) >= p:
            return k
    return classes

In [ ]:
print(pbirthday(23))    # probability of at least one match with 23 people
print(qbirthday(0.5))   # how many people for a 50% chance?

`pbirthday(23)` returns approximately 0.507 and `qbirthday(0.5)` returns 23, confirming our calculation.

We can also find the probability of having at least one *triple* birthday match — three people with the same birthday — by adding `coincident=3`. For example, `pbirthday(23, coincident=3)` returns approximately 0.014, so 23 people give us only about a 1.4% chance of a triple birthday match. `qbirthday(0.5, coincident=3)` returns 88, so we would need 88 people to have at least a 50% chance of a triple match.

In [ ]:
print(pbirthday(23, coincident=3))    # P(triple match | 23 people) ~0.014
print(qbirthday(0.5, coincident=3))   # need ~88 people for 50% chance of triple match

To simulate the birthday problem, we can generate random birthdays for 23 people and then count how many people share each day using `np.bincount`:

In [ ]:
b = rng.integers(1, 366, size=23)   # random birthdays for 23 people (days 1..365)
np.bincount(b)[1:]                  # count how many people share each day

We can run 10 000 repetitions as follows:

In [ ]:
r = [np.bincount(rng.integers(0, 365, size=23)).max()   # max count on any single day
     for _ in range(10**4)]
np.sum(np.array(r) >= 2) / 10**4                        # proportion with at least one shared birthday

If the probabilities of various days are not all equal, the calculation becomes much more difficult, but the simulation can easily be extended since `rng.choice` allows us to specify the probability of each day via the `p` argument (by default `rng.integers` assigns equal probabilities, so above the probability is 1/365 for each day).